# Clip **local** 2018 Alpha Earth raster → LandIQ **2018** tomato polygons

**Input polygons:** GeoPackage from [`../../../landiq/years/2018/02_filter_tomato_polygons.ipynb`](../../../landiq/years/2018/02_filter_tomato_polygons.ipynb) (path from `landiq_tomato_gpkg_path` in `configs/paths.local.yaml`, usually `landiq_tomato_2018.gpkg`).

**Input raster:** One **GeoTIFF** (single- or multi-band) for **2018** under:

`data/derived/alpha_earth_rasters/2018/` (repo-relative: `alpha_earth.local_raster_root` + `/2018/`).

The helper picks the first `*.tif` in that folder (shallow, then recursive). Export from Earth Engine, GCS, or your vendor stack into that layout.

**Output:** One GeoTIFF clipped to the **union** of all tomato geometries → `data/derived/alpha_earth_clips/local/landiq2018/` (gitignored).

**Earth Engine** per-polygon downloads instead: [`../../earth_engine/years/2018/01_pilot_landiq2018_alphaearth_ee.ipynb`](../../earth_engine/years/2018/01_pilot_landiq2018_alphaearth_ee.ipynb).

In [ ]:
from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import yaml

from src.alpha_earth.clip_to_polygons import clip_raster_to_gdf, resolve_raster_path_for_year
from src.utils.paths import REPO_ROOT, landiq_tomato_gpkg_path

SURVEY_YEAR = 2018

cfg_path = REPO_ROOT / "configs" / "paths.local.yaml"
if not cfg_path.is_file():
    cfg_path = REPO_ROOT / "configs" / "paths.example.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

ly = cfg.get("landiq", {}).get("year")
if ly is not None and int(ly) != SURVEY_YEAR:
    print(f"Note: landiq.year={ly} but this notebook is for {SURVEY_YEAR}; align config or change SURVEY_YEAR.")

ae = cfg.get("alpha_earth", {})
raster_root = REPO_ROOT / ae.get("local_raster_root", "data/derived/alpha_earth_rasters")
clips_base = REPO_ROOT / cfg["data"]["alpha_earth_clips"] / "local" / f"landiq{SURVEY_YEAR}"

tomato_path = landiq_tomato_gpkg_path(cfg, REPO_ROOT)
if not tomato_path.is_file():
    raise FileNotFoundError(f"Tomato GPKG missing: {tomato_path}. Run LandIQ filter notebook first.")

raster_path = resolve_raster_path_for_year(raster_root, SURVEY_YEAR)
if raster_path is None:
    raise FileNotFoundError(
        f"No GeoTIFF for {SURVEY_YEAR} under {raster_root.resolve()}. "
        f"Create folder {raster_root / str(SURVEY_YEAR)} and place a .tif there."
    )

polygons = gpd.read_file(tomato_path)
print("Tomato GPKG:", tomato_path.resolve())
print("Polygons:", len(polygons))
print("2018 raster:", raster_path.resolve())
print("Output dir:", clips_base.resolve())

In [ ]:
import rasterio

out_name = f"clip_{SURVEY_YEAR}_tomato_union__{raster_path.stem}.tif"
out_path = clips_base / out_name

with rasterio.open(raster_path) as ds:
    clip_raster_to_gdf(ds, polygons, out_path)

print("Wrote:", out_path.resolve())

### Quick QC

In [ ]:
import rasterio

with rasterio.open(out_path) as ds:
    print("Bands, height, width:", ds.count, ds.height, ds.width)
    print("CRS:", ds.crs)